# Methane Detection from Hyperspectral Imagery (H-MRCNN)
### Google Colab / Kaggle GPU Execution Pipeline

This notebook runs the complete end-to-end pipeline:
1. **Environment Setup**: Configures TensorFlow 1.15 + Keras 2.2.4 on Python 3.7 with GPU support.
2. **Pre-trained Weights**: Automatically downloads MS-COCO pre-trained Mask R-CNN weights.
3. **Data Preprocessing**: 
   - Mask Tiling (`mask_tiling.py`)
   - Contour & Annotation JSON Extraction (`json_creator.py`)
   - Radiance Tile Generation (`train_data_generator.py`)
4. **Model Training**: Fine-tunes Mask R-CNN on hyperspectral methane plume data.
5. **Prediction & Visualization**: Generates detections and visualizes results inline.

> **Important**: Make sure GPU is enabled in Google Colab: **Runtime -> Change runtime type -> T4 GPU**.

--- 
### Step 1: Verify GPU Allocation

In [ ]:
!nvidia-smi

--- 
### Step 2: Clone or Mount Workspace
If running directly in Google Colab, you can mount Google Drive (recommended if your dataset is in Drive) or clone the repository.

In [ ]:
import os

# If using Google Drive to store/load large hyperspectral datasets:
# from google.colab import drive
# drive.mount('/content/drive')

# Clone the repository if not already cloned in current directory
if not os.path.exists('Methane-detection-from-hyperspectral-imagery'):
    !git clone https://github.com/satish1901/Methane-detection-from-hyperspectral-imagery.git

%cd Methane-detection-from-hyperspectral-imagery
!pwd

--- 
### Step 3: Install Compatible Environment (Python 3.7 + TF 1.15 + Keras 2.2.4)
The codebase requires TensorFlow 1.x and Keras 2.2.4. We use `condacolab` to set up a clean Python 3.7 environment.

In [ ]:
# Install CondaColab to get conda on Google Colab
!pip install -q condacolab
import condacolab
condacolab.install()

> **Note**: The Colab session may restart after running `condacolab.install()`. This is expected. Continue from the next cell.

In [ ]:
# Install Python 3.7 packages compatible with Mask R-CNN & TF 1.x
!conda install -y python=3.7 pip

# Install specific compatible wheels:
# Notice h5py is pinned to 2.10.0 because h5py 3.x breaks Keras 2.2 weights loading
!pip install tensorflow-gpu==1.15.0
!pip install keras==2.2.4
!pip install h5py==2.10.0
!pip install numpy==1.19.5
!pip install scipy==1.4.1
!pip install scikit-image==0.16.2
!pip install opencv-python-headless
!pip install spectral==0.22.4
!pip install imutils==0.5.4
!pip install coloredlogs==15.0.1
!pip install matplotlib

--- 
### Step 4: Verify TensorFlow and GPU Availability in Python

In [ ]:
import tensorflow as tf
import keras
print("TensorFlow Version:", tf.__version__)
print("Keras Version:", keras.__version__)
print("Is GPU available?", tf.test.is_gpu_available())
print("GPU Device Name:", tf.test.gpu_device_name())

--- 
### Step 5: Download Pretrained COCO Mask R-CNN Weights
The detector initializes from `mask_rcnn_coco.h5` for transfer learning.

In [ ]:
import os
import urllib.request

coco_weights_dir = "single_detector/src/custom-mask-rcnn-detector"
coco_weights_path = os.path.join(coco_weights_dir, "mask_rcnn_coco.h5")

if not os.path.exists(coco_weights_path):
    print("Downloading mask_rcnn_coco.h5...")
    url = "https://github.com/matterport/Mask_RCNN/releases/download/v2.0/mask_rcnn_coco.h5"
    urllib.request.urlretrieve(url, coco_weights_path)
    print("Downloaded successfully to", coco_weights_path)
else:
    print("Weights file already exists:", coco_weights_path)

--- 
### Step 6: Dataset Setup

Place your flight data folder(s) inside `single_detector/data/<flight_id>/` with:
- `<flight_id>_img` (hyperspectral ENVI binary)
- `<flight_id>_img.hdr` (ENVI header)
- `<flight_id>_img_mask.png` (ground truth mask)

*Optional*: If you want to test the entire pipeline end-to-end first before loading your full flight dataset, run the cell below to generate synthetic flight data.

In [ ]:
# Generate synthetic sample flight data for testing (optional)
!python single_detector/data/create_dummy_flight_data.py

--- 
### Step 7: Full Data Preprocessing Pipeline

This runs the 3-stage preprocessing:
1. **`mask_tiling.py`**: Reads binary mask (`*_img_mask.png`) and breaks it into 256x256 tiles.
2. **`json_creator.py`**: Traces methane plume boundaries using DFS and writes `annotation_plumes.json`.
3. **`train_data_generator.py`**: Reads hyperspectral data (`*_img` & `*_img.hdr`), applies local and global normalization, and creates 3-channel tiles for Mask R-CNN.

In [ ]:
# Execute the full data preprocessing generator script
# Arguments: path to data directory
!cd single_detector && chmod +x data_generator.sh && ./data_generator.sh '../data'

--- 
### Step 8: Train the Single-Detector Model

We add the custom `mask_rcnn` module to `PYTHONPATH` and start training `detector.py`.

In [ ]:
import os
# Set PYTHONPATH so mrcnn module is found
repo_root = os.path.abspath(".")
mrcnn_path = os.path.join(repo_root, "single_detector", "src", "custom-mask-rcnn-detector", "mask_rcnn")
os.environ['PYTHONPATH'] = f"{mrcnn_path}:{os.environ.get('PYTHONPATH', '')}"

# Run training
!cd single_detector/src/custom-mask-rcnn-detector && python detector.py --mode train

--- 
### Step 9: Run Model Inference and Plume Detection

In [ ]:
import glob
import os

# Find a sample tile from the generated train_data to test inference
sample_tiles = glob.glob("single_detector/src/custom-mask-rcnn-detector/ch4_data/train_data/*/*.npy")

if sample_tiles:
    test_img = sample_tiles[0]
    print("Running inference on:", test_img)
    !cd single_detector/src/custom-mask-rcnn-detector && python detector.py --mode predict --image "{os.path.abspath(test_img)}"
else:
    print("No .npy test tile found. Please run preprocessing first.")

--- 
### Step 10: Visualize Predicted Methane Plume Masks

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

# Find saved prediction outputs
output_images = glob.glob("single_detector/src/custom-mask-rcnn-detector/*output*.png") + glob.glob("single_detector/src/custom-mask-rcnn-detector/*.png")

if output_images:
    plt.figure(figsize=(12, 8))
    img = mpimg.imread(output_images[0])
    plt.imshow(img)
    plt.axis('off')
    plt.title('Detected Methane Plume')
    plt.show()
else:
    print("Run inference step to generate visual outputs.")